In [3]:
"""
NoBroker Property Data Extractor
---------------------------------
Reads input.csv (columns: id, title, link, date, savedAt)
Extracts property data from each NoBroker listing URL
Saves to final_excel.xlsx (skips already-extracted records)

Requirements:
    pip install selenium openpyxl pandas webdriver-manager
"""

import os
import sys
import time
import logging
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ── Config ─────────────────────────────────────────────────────────────────────
INPUT_CSV    = "input.csv"
OUTPUT_EXCEL = "final_excel.xlsx"
LOGIN_WAIT   = 40          # seconds to wait for manual login
PAGE_WAIT    = 15          # seconds to wait for a listing page to load

# ── Columns (must match JS extraction order exactly) ────────────────────────────
COLUMNS = [
    "id", "url", "status", "phone", "visitDate",
    "title", "location", "rent", "maintenance", "area", "deposit",
    "bedrooms", "propertyType", "preferredTenant", "possession",
    "parking", "buildingAge", "balcony", "postedOn",
    "furnishingStatus", "facing", "waterSupply", "floor", "bathroom",
    "petAllowed", "nonVegAllowed", "gatedSecurity",
    "latitude", "longitude",
    "uniqueViews", "shortlists", "contacted",
    "comments"
]

# ── Logging setup ───────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("nobroker_scraper.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)

# ── JavaScript extraction (mirrors the JS bookmarklet) ─────────────────────────
EXTRACT_JS = """
function extractIdFromUrl(url) {
    try {
        var match = url.match(/\\/property\\/[^\\/]+\\/([a-f0-9]+)\\/detail/i);
        if (match && match[1]) return match[1];
        var parts = url.split('/');
        var detailIndex = parts.indexOf('detail');
        if (detailIndex > 0) return parts[detailIndex - 1];
        return '';
    } catch(e) { return ''; }
}

function safeExtract(selector, attribute) {
    try {
        attribute = attribute || 'textContent';
        var el = document.querySelector(selector);
        if (!el) return '';
        if (attribute === 'textContent') return el.textContent.trim().replace(/\\s+/g, ' ');
        if (attribute === 'content')     return el.getAttribute('content') || '';
        return el.getAttribute(attribute) || '';
    } catch(e) { return ''; }
}

function extractById(id) {
    try {
        var el = document.getElementById(id);
        return el ? el.textContent.trim().replace(/\\s+/g, ' ') : '';
    } catch(e) { return ''; }
}

function extractOverviewItem(title) {
    try {
        var items = document.querySelectorAll('.nb__3ocPe');
        for (var i = 0; i < items.length; i++) {
            var titleEl = items[i].querySelector('.nb__1IoiM');
            if (titleEl && titleEl.textContent.includes(title)) {
                var valueEl = items[i].querySelector('.font-semi-bold');
                return valueEl ? valueEl.textContent.trim() : '';
            }
        }
        return '';
    } catch(e) { return ''; }
}

var data = {
    id:               extractIdFromUrl(window.location.href),
    url:              window.location.href,
    status:           'Not started',
    phone:            '',
    visitDate:        '',
    title:            safeExtract('h1.nb__s_YQN'),
    location:         safeExtract('h5.nb__16pur'),
    rent:             safeExtract('#rent-maintenance span[itemprop="value"] > div > div > span') ||
                      safeExtract('#rent-maintenance .flex.gap-1 > span') ||
                      safeExtract('#rent-maintenance .nb__3h7Fo > span[itemprop="value"]'),
    maintenance:      safeExtract('#rent-maintenance .text-14.font-semibold.ml-1'),
    area:             safeExtract('#square-ft'),
    deposit:          safeExtract('#emi span[itemprop="value"]') || safeExtract('#emi'),
    bedrooms:         extractById('details-summary-typeDesc'),
    propertyType:     extractById('details-summary-buildingType'),
    preferredTenant:  extractById('details-summary-leaseType'),
    possession:       extractById('details-summary-availableFrom'),
    parking:          extractById('details-summary-parkingDesc'),
    buildingAge:      extractById('details-summary-propertyAge'),
    balcony:          extractById('details-summary-balconies'),
    postedOn:         extractById('details-summary-lastUpdateDate'),
    furnishingStatus: extractOverviewItem('Furnishing Status'),
    facing:           extractOverviewItem('Facing'),
    waterSupply:      extractOverviewItem('Water Supply'),
    floor:            extractOverviewItem('Floor'),
    bathroom:         extractOverviewItem('Bathroom'),
    petAllowed:       extractOverviewItem('Pet Allowed'),
    nonVegAllowed:    extractOverviewItem('Non-Veg Allowed'),
    gatedSecurity:    extractOverviewItem('Gated Security'),
    latitude:         safeExtract('meta[itemprop="latitude"]',  'content'),
    longitude:        safeExtract('meta[itemprop="longitude"]', 'content'),
    uniqueViews:      '',
    shortlists:       '',
    contacted:        '',
    comments:         ''
};

// Activity metrics
try {
    var actDivs = document.querySelectorAll('.nb__3PMUu');
    if (actDivs.length >= 1) data.uniqueViews  = (actDivs[0].querySelector('.nb__32dA0') || {}).textContent || '';
    if (actDivs.length >= 2) data.shortlists   = (actDivs[1].querySelector('.nb__32dA0') || {}).textContent || '';
    if (actDivs.length >= 3) data.contacted    = (actDivs[2].querySelector('.nb__32dA0') || {}).textContent || '';
} catch(e) {}

return data;
"""


# ── Excel helpers ───────────────────────────────────────────────────────────────

def load_or_create_excel(path: str) -> pd.DataFrame:
    if os.path.exists(path):
        log.info(f"Loading existing Excel: {path}")
        df = pd.read_excel(path, dtype=str)
        # Ensure all expected columns exist
        for col in COLUMNS:
            if col not in df.columns:
                df[col] = ""
        return df[COLUMNS]
    else:
        log.info(f"final_excel.xlsx not found – creating empty workbook: {path}")
        df = pd.DataFrame(columns=COLUMNS)
        df.to_excel(path, index=False)
        return df


def save_excel(df: pd.DataFrame, path: str):
    df.to_excel(path, index=False)
    log.info(f"Saved {len(df)} records → {path}")


# ── Selenium helpers ────────────────────────────────────────────────────────────

def build_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


def wait_for_page(driver: webdriver.Chrome, timeout: int = PAGE_WAIT):
    """Wait until page readyState is complete."""
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    # Extra buffer for JS-rendered content
    time.sleep(3)


def extract_property(driver: webdriver.Chrome, url: str) -> dict:
    log.info(f"  ↳ Navigating to: {url}")
    driver.get(url)
    wait_for_page(driver)
    data = driver.execute_script(EXTRACT_JS)
    # Ensure url matches the input url (handles redirects)
    data["url"] = url
    return data


# ── Main ────────────────────────────────────────────────────────────────────────

def main():
    # 1. Read input CSV
    if not os.path.exists(INPUT_CSV):
        log.error(f"Input file not found: {INPUT_CSV}")
        sys.exit(1)

    input_df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")
    log.info(f"Input CSV loaded: {len(input_df)} rows")

    if "link" not in input_df.columns:
        log.error("'link' column not found in input.csv")
        sys.exit(1)

    # 2. Load / create output Excel
    out_df = load_or_create_excel(OUTPUT_EXCEL)

    # Build set of already-extracted URLs for fast lookup
    existing_urls = set(out_df["url"].dropna().str.strip().tolist())
    log.info(f"Already extracted: {len(existing_urls)} records")

    # 3. Determine which rows need extraction
    pending = [
        row for _, row in input_df.iterrows()
        if str(row.get("link", "")).strip() not in existing_urls
    ]
    skipped = len(input_df) - len(pending)
    log.info(f"Skipping {skipped} already-extracted records")
    log.info(f"Pending extraction: {len(pending)} records")

    if not pending:
        log.info("Nothing to do – all records already extracted.")
        return

    # 4. Launch browser and wait for login
    log.info("Launching Chrome browser…")
    driver = build_driver()

    try:
        log.info("Opening NoBroker homepage…")
        driver.get("https://www.nobroker.in")
        log.info(f"⏳  Please log in within the next {LOGIN_WAIT} seconds…")

        for remaining in range(LOGIN_WAIT, 0, -5):
            log.info(f"   Waiting {remaining}s for login…")
            time.sleep(5)

        log.info("Proceeding with extraction…")

        # 5. Extract each pending property
        new_rows = []
        total   = len(pending)

        for idx, row in enumerate(pending, start=1):
            url = str(row.get("link", "")).strip()
            log.info(f"[{idx}/{total}] Processing: {url}")

            try:
                data = extract_property(driver, url)

                # Log key fields
                log.info(f"  ✔  title='{data.get('title','')}'  "
                         f"rent='{data.get('rent','')}'  "
                         f"location='{data.get('location','')}'")

                # Build record in COLUMNS order
                record = {col: str(data.get(col, "") or "").strip() for col in COLUMNS}
                new_rows.append(record)

                # Save after every successful extraction (crash-safe)
                new_df = pd.DataFrame(new_rows, columns=COLUMNS)
                out_df = pd.concat([out_df, new_df.iloc[[-1]]], ignore_index=True)
                save_excel(out_df, OUTPUT_EXCEL)

            except Exception as e:
                log.error(f"  ✗  Failed to extract {url}: {e}")
                # Still record the URL with empty fields so we can inspect later
                record = {col: "" for col in COLUMNS}
                record["url"]    = url
                record["status"] = "ERROR"
                new_rows.append(record)
                out_df = pd.concat([out_df, pd.DataFrame([record], columns=COLUMNS)],
                                   ignore_index=True)
                save_excel(out_df, OUTPUT_EXCEL)

            # Polite delay between requests
            time.sleep(2)

        log.info(f"Extraction complete. {len(new_rows)} new records added.")
        log.info(f"Final file: {OUTPUT_EXCEL}  ({len(out_df)} total rows)")

    finally:
        driver.quit()
        log.info("Browser closed.")


if __name__ == "__main__":
    main()

2026-03-03 03:51:59,596  INFO      Input CSV loaded: 4 rows
2026-03-03 03:51:59,597  INFO      Loading existing Excel: final_excel.xlsx
2026-03-03 03:51:59,640  INFO      Already extracted: 4 records
2026-03-03 03:51:59,651  INFO      Skipping 4 already-extracted records
2026-03-03 03:51:59,657  INFO      Pending extraction: 0 records
2026-03-03 03:51:59,668  INFO      Nothing to do – all records already extracted.
